# Notebook 2: Text Transfer Learning — RoBERTa vs ModernBERT + Confidence Calibration

This notebook investigates text classification via transfer learning on emotion detection, with a deep focus on model **confidence calibration** — a topic of critical importance for deployment but often overlooked in academic benchmarking.

**Models compared:**
- **DistilBERT** — reference baseline (66M params, 40% smaller/faster than BERT)
- **RoBERTa-base** — robustly optimized BERT (125M params, no NSP, dynamic masking)
- **ModernBERT-base** — 2024 re-architecture with RoPE, Flash Attention, and ALiBi (150M params)

**Key questions:**
1. Does ModernBERT's architectural modernization yield better accuracy on short emotional texts?
2. Are these models well-calibrated? Does high accuracy imply calibrated confidence?
3. Can temperature scaling reliably reduce ECE post-hoc without retraining?
4. What does ModernBERT's longer context window actually change for this dataset?

## Section 0: Imports & Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import torch, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from configs.text_config import TEXT_MODELS, EMOTION_CLASSES, TextTrainingConfig
from src.text.trainer import train_text_model
from src.utils.metrics import compute_ece, TemperatureScaler, compute_classification_metrics
from src.utils.visualization import plot_confusion_matrix, plot_reliability_diagram
import json
import pandas as pd
from pathlib import Path
from PIL import Image

RESULTS_PATH = Path("../results/text")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)
Path("../results/figures").mkdir(parents=True, exist_ok=True)

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"Emotion classes: {EMOTION_CLASSES}")

## Section 1: Dataset Exploration — dair-ai/emotion

The **dair-ai/emotion** dataset (Saravia et al., 2018) contains Twitter messages labeled with one of six basic emotions:
`sadness`, `joy`, `love`, `anger`, `fear`, `surprise`.

It has **16,000 training** / **2,000 validation** / **2,000 test** examples. The texts are short (~15 words average), written in informal Twitter language with abbreviations, slang, and emojis — making it a realistic NLP benchmark.

In [ ]:
dataset = load_dataset("dair-ai/emotion")
train_ds = dataset["train"]
val_ds   = dataset["validation"]
test_ds  = dataset["test"]

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")
print(f"Columns: {train_ds.column_names}")
print(f"\nLabel mapping: {train_ds.features['label'].names}")

In [ ]:
# Class distribution
from collections import Counter

label_names = train_ds.features["label"].names
counts = Counter(train_ds["label"])
sorted_counts = [(label_names[k], counts[k]) for k in sorted(counts)]

fig, ax = plt.subplots(figsize=(9, 4))
emotion_palette = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f", "#edc948"]
classes, freqs = zip(*sorted_counts)
bars = ax.bar(classes, freqs, color=emotion_palette, edgecolor="white", linewidth=0.8)
ax.set_title("dair-ai/emotion Training Set — Class Distribution", fontsize=13, fontweight="bold")
ax.set_xlabel("Emotion Class")
ax.set_ylabel("Number of Examples")
for bar, count in zip(bars, freqs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f"{count:,}", ha="center", va="bottom", fontsize=9, fontweight="bold")

max_count = max(freqs)
min_count = min(freqs)
imbalance_ratio = max_count / min_count
ax.text(0.98, 0.95, f"Imbalance ratio: {imbalance_ratio:.1f}×",
        transform=ax.transAxes, ha="right", va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#fff3cd", edgecolor="#ffc107"))

plt.tight_layout()
plt.savefig("../results/figures/emotion_class_distribution.png", dpi=150)
plt.show()
print(f"Class imbalance: max={max_count:,} ({label_names[max(counts, key=counts.get)]}), min={min_count:,} ({label_names[min(counts, key=counts.get)]})")

In [ ]:
# Display 5 example tweets per emotion class
class_examples = {label: [] for label in range(len(label_names))}
for sample in train_ds:
    lbl = sample["label"]
    if len(class_examples[lbl]) < 5:
        class_examples[lbl].append(sample["text"])

print("Sample tweets per emotion class:\n")
for cls_idx, examples in class_examples.items():
    print(f"{'='*60}")
    print(f"  Emotion: {label_names[cls_idx].upper()}")
    print(f"{'='*60}")
    for i, text in enumerate(examples, 1):
        print(f"  {i}. {text[:120]}")
    print()

## Section 2: Model Comparison

| Model | Year | Params | Architecture Changes vs BERT | Key Innovation |
|-------|------|--------|------------------------------|----------------|
| **DistilBERT** | 2019 | 66M | Knowledge distillation from BERT | 40% fewer params, 60% faster, retains 97% BERT performance |
| **RoBERTa-base** | 2019 | 125M | Removed NSP objective, dynamic masking, more data/longer training | Robustly optimized — shows NSP was hurting BERT |
| **ModernBERT-base** | 2024 | 150M | **RoPE** (Rotary Positional Embedding), **Flash Attention 2**, **ALiBi**, unpadded sequences, global+local attention | State-of-the-art efficiency; extends to 8192 context |

### ModernBERT key architectural innovations:
- **RoPE (Rotary Positional Embedding)**: Better length generalization than absolute positional embeddings used in BERT/RoBERTa. Allows extending context without performance degradation.
- **Flash Attention 2**: IO-aware attention algorithm that is 2-4× faster than standard attention and uses 5-20× less memory — enables longer sequences and larger batches.
- **Alternating global/local attention**: Local attention layers (sliding window) for most layers + global attention every 3rd layer. Dramatically reduces quadratic attention cost for long sequences.

## Section 3: Training — RoBERTa and ModernBERT

In [ ]:
training_results_file = RESULTS_PATH / "training_results.json"

if training_results_file.exists():
    print("Loading pre-computed training results...")
    with open(training_results_file) as f:
        training_results = json.load(f)
else:
    print("Training RoBERTa and ModernBERT on dair-ai/emotion...")
    training_results = {}

    for model_key in ["roberta", "modernbert"]:
        print(f"\n  Training {model_key}...")
        cfg = TextTrainingConfig(model_key=model_key)
        results = train_text_model(cfg)
        training_results[model_key] = results
        print(f"    → val_acc={results['val_acc'][-1]:.4f}, test_acc={results['test_acc']:.4f}")

    with open(training_results_file, "w") as f:
        json.dump(training_results, f, indent=2)
    print("\nResults saved.")

In [ ]:
# Side-by-side validation accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training Dynamics — RoBERTa vs ModernBERT on dair-ai/emotion",
             fontsize=13, fontweight="bold")

colors = {"roberta": "#4e79a7", "modernbert": "#e15759"}

for ax, model_key in zip(axes, ["roberta", "modernbert"]):
    res = training_results[model_key]
    epochs = range(1, len(res["val_acc"]) + 1)
    ax.plot(epochs, [v * 100 for v in res["train_acc"]], "--",
            color=colors[model_key], alpha=0.7, label="Train Acc")
    ax.plot(epochs, [v * 100 for v in res["val_acc"]], "-",
            color=colors[model_key], linewidth=2, label="Val Acc")
    ax.axhline(res["test_acc"] * 100, color="gray", linestyle=":",
               linewidth=1.5, label=f"Test Acc: {res['test_acc']*100:.2f}%")
    ax.set_title(f"{TEXT_MODELS[model_key]['display_name']}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy (%)")
    ax.legend(fontsize=9)
    ax.set_ylim(50, 100)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("../results/figures/text_training_curves.png", dpi=150)
plt.show()

print("\nFinal Results:")
for m in ["roberta", "modernbert"]:
    r = training_results[m]
    print(f"  {m:<15}: test_acc={r['test_acc']*100:.2f}%, f1_macro={r.get('f1_macro', 0)*100:.2f}%")

## Section 4: Confidence Calibration Study

### What is calibration and why does it matter?

A model is **perfectly calibrated** if, among all predictions it makes with confidence 70%, exactly 70% turn out to be correct. Formally, for confidence value $\hat{p}$:

$$\mathbb{P}(\hat{Y} = Y \mid \hat{P} = \hat{p}) = \hat{p}, \quad \forall \hat{p} \in [0, 1]$$

### Expected Calibration Error (ECE)

**Guo et al. (2017)** — *"On Calibration of Modern Neural Networks"* showed that modern deep networks are systematically **overconfident**: they produce high confidence scores even for incorrect predictions. They proposed ECE as the standard metric:

$$\text{ECE} = \sum_{m=1}^{M} \frac{|B_m|}{n} \left| \text{acc}(B_m) - \text{conf}(B_m) \right|$$

Where $B_m$ are confidence bins, $\text{acc}(B_m)$ is the accuracy within the bin, and $\text{conf}(B_m)$ is the mean confidence within the bin. Lower ECE = better calibration.

### Temperature Scaling

**Temperature scaling** is the simplest post-hoc calibration method: divide logits by a learned scalar $T > 0$ before softmax. No accuracy change, only confidence distribution shifts. Fit $T$ on the validation set using NLL minimization:

$$\hat{y}_i = \text{softmax}(z_i / T)$$

A temperature $T > 1$ softens the distribution (reduces overconfidence); $T < 1$ sharpens it.

In [ ]:
# Compute ECE before and after temperature scaling for each model
calibration_results = {}

for model_key in ["roberta", "modernbert"]:
    print(f"\nCalibrating {model_key}...")

    model_cfg = TEXT_MODELS[model_key]
    tokenizer = AutoTokenizer.from_pretrained(model_cfg["model_id"])
    model = AutoModelForSequenceClassification.from_pretrained(
        model_cfg["model_id"],
        num_labels=len(EMOTION_CLASSES)
    )

    # Load best checkpoint
    ckpt = RESULTS_PATH / model_key / "best_model.pt"
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt, map_location="cpu"))
    model.eval()

    # Collect val logits and test logits
    def get_logits_and_labels(split_ds):
        all_logits, all_labels = [], []
        batch_size = 32
        for i in range(0, len(split_ds), batch_size):
            batch = split_ds[i:i + batch_size]
            enc = tokenizer(batch["text"], truncation=True, padding=True,
                            max_length=model_cfg.get("max_length", 128), return_tensors="pt")
            with torch.no_grad():
                out = model(**enc)
            all_logits.append(out.logits.cpu())
            all_labels.extend(batch["label"])
        return torch.cat(all_logits), torch.tensor(all_labels)

    val_logits, val_labels   = get_logits_and_labels(val_ds)
    test_logits, test_labels = get_logits_and_labels(test_ds)

    # ECE before calibration
    test_probs_before = torch.softmax(test_logits, dim=-1).numpy()
    ece_before = compute_ece(test_probs_before, test_labels.numpy())

    # Fit temperature on validation set
    scaler = TemperatureScaler()
    scaler.fit(val_logits, val_labels)
    T = scaler.temperature.item()

    # ECE after calibration
    test_probs_after = scaler.calibrate(test_logits).numpy()
    ece_after = compute_ece(test_probs_after, test_labels.numpy())

    # Accuracy and F1
    preds = np.argmax(test_probs_before, axis=1)
    metrics = compute_classification_metrics(preds, test_labels.numpy(), num_classes=len(EMOTION_CLASSES))

    calibration_results[model_key] = {
        "test_acc": metrics["accuracy"],
        "f1_macro": metrics["f1_macro"],
        "ece_before": ece_before,
        "ece_after": ece_after,
        "temperature": T,
        "val_logits": val_logits,
        "val_labels": val_labels,
        "test_logits": test_logits,
        "test_labels": test_labels,
        "test_probs_before": test_probs_before,
        "test_probs_after": test_probs_after,
    }

    print(f"  T={T:.3f} | ECE before={ece_before:.4f} | ECE after={ece_after:.4f}")

In [ ]:
# Reliability diagrams
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle("Reliability Diagrams — Before and After Temperature Scaling",
             fontsize=13, fontweight="bold")

for row, model_key in enumerate(["roberta", "modernbert"]):
    res = calibration_results[model_key]

    # Before calibration
    plot_reliability_diagram(
        probs=res["test_probs_before"],
        labels=res["test_labels"].numpy(),
        ax=axes[row][0],
        title=f"{TEXT_MODELS[model_key]['display_name']} — Before (ECE={res['ece_before']:.4f})",
        n_bins=10
    )

    # After calibration
    plot_reliability_diagram(
        probs=res["test_probs_after"],
        labels=res["test_labels"].numpy(),
        ax=axes[row][1],
        title=f"{TEXT_MODELS[model_key]['display_name']} — After T={res['temperature']:.3f} (ECE={res['ece_after']:.4f})",
        n_bins=10
    )

plt.tight_layout()
plt.savefig("../results/figures/reliability_diagrams.png", dpi=150)
plt.show()

In [ ]:
# Calibration comparison table
print("\nCalibration Summary Table")
print("=" * 80)
print(f"{'Model':<18} {'Test Acc':>10} {'F1 Macro':>10} {'ECE Before':>12} {'ECE After':>11} {'Temp T':>8}")
print("-" * 80)

for model_key in ["roberta", "modernbert"]:
    res = calibration_results[model_key]
    name = TEXT_MODELS[model_key]["display_name"]
    print(f"{name:<18} {res['test_acc']*100:>9.2f}% {res['f1_macro']*100:>9.2f}% "
          f"{res['ece_before']:>12.4f} {res['ece_after']:>11.4f} {res['temperature']:>8.3f}")

print("\nInterpretation:")
for model_key in ["roberta", "modernbert"]:
    res = calibration_results[model_key]
    improvement = (res["ece_before"] - res["ece_after"]) / res["ece_before"] * 100
    name = TEXT_MODELS[model_key]["display_name"]
    print(f"  {name}: Temperature scaling reduced ECE by {improvement:.1f}% (T={res['temperature']:.3f})")

## Section 5: Error Analysis

In [ ]:
# Best model confusion matrix (select by test accuracy)
best_model_key = max(["roberta", "modernbert"],
                     key=lambda m: calibration_results[m]["test_acc"])
print(f"Best model: {best_model_key} ({calibration_results[best_model_key]['test_acc']*100:.2f}%)")

best_res = calibration_results[best_model_key]
best_preds = np.argmax(best_res["test_probs_before"], axis=1)
best_labels = best_res["test_labels"].numpy()

plot_confusion_matrix(
    y_true=best_labels,
    y_pred=best_preds,
    class_names=EMOTION_CLASSES,
    normalize=True,
    title=f"{TEXT_MODELS[best_model_key]['display_name']} — Emotion Classification Confusion Matrix",
    save_path="../results/figures/text_confusion_matrix.png"
)

In [ ]:
# Per-class precision/recall bar chart
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    best_labels, best_preds, labels=list(range(len(EMOTION_CLASSES)))
)

x = np.arange(len(EMOTION_CLASSES))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - width/2, precision * 100, width, label="Precision", color="#4e79a7", edgecolor="white")
ax.bar(x + width/2, recall * 100, width, label="Recall", color="#e15759", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(EMOTION_CLASSES, fontsize=10)
ax.set_ylabel("Score (%)")
ax.set_title(f"{TEXT_MODELS[best_model_key]['display_name']} — Per-Class Precision & Recall",
             fontsize=12, fontweight="bold")
ax.legend()
ax.set_ylim(0, 105)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

for i, (p, r) in enumerate(zip(precision, recall)):
    ax.text(i - width/2, p * 100 + 1, f"{p*100:.0f}", ha="center", fontsize=7)
    ax.text(i + width/2, r * 100 + 1, f"{r*100:.0f}", ha="center", fontsize=7)

plt.tight_layout()
plt.savefig("../results/figures/per_class_precision_recall.png", dpi=150)
plt.show()

In [ ]:
# 10 hardest misclassified examples — highest confidence wrong predictions
test_texts = test_ds["text"]
probs = best_res["test_probs_before"]
max_confs = probs.max(axis=1)

# Find wrong predictions sorted by descending confidence
wrong_mask = best_preds != best_labels
wrong_indices = np.where(wrong_mask)[0]
wrong_confs = max_confs[wrong_indices]
top_10_idx = wrong_indices[np.argsort(-wrong_confs)[:10]]

print("Top 10 Most Confidently Wrong Predictions\n")
print(f"{'#':<4} {'Confidence':>12} {'True':>12} {'Predicted':>12}  {'Text'}")
print("-" * 100)

for rank, idx in enumerate(top_10_idx, 1):
    conf = max_confs[idx]
    true_cls = EMOTION_CLASSES[best_labels[idx]]
    pred_cls = EMOTION_CLASSES[best_preds[idx]]
    text = test_texts[idx][:70]
    print(f"{rank:<4} {conf*100:>11.1f}% {true_cls:>12} {pred_cls:>12}  {text}...")

## Section 6: ModernBERT vs RoBERTa — Tokenisation Analysis

### What does ModernBERT's longer context window actually change?

ModernBERT supports up to **8,192 tokens** per input (vs RoBERTa's 512). For the emotion dataset, which consists of tweets averaging ~15 words, this difference is irrelevant for accuracy. However, this section illustrates an important point:

**The two tokenizers produce different token counts for the same text** because they use different vocabularies and subword splitting strategies:
- **RoBERTa** uses Byte-Pair Encoding (BPE) with a 50,265-token vocabulary
- **ModernBERT** uses an updated tokenizer with a 50,368-token vocabulary and different merge rules

For informal Twitter text (abbreviations, slang, emojis), this can cause significant differences in tokenization, affecting how the model processes the input.

In [ ]:
# Load tokenizers for both models
roberta_tokenizer   = AutoTokenizer.from_pretrained(TEXT_MODELS["roberta"]["model_id"])
modernbert_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODELS["modernbert"]["model_id"])

# Compute token count distributions
sample_texts = train_ds["text"]

roberta_counts    = [len(roberta_tokenizer.encode(t, add_special_tokens=True)) for t in sample_texts]
modernbert_counts = [len(modernbert_tokenizer.encode(t, add_special_tokens=True)) for t in sample_texts]

print(f"RoBERTa    — mean: {np.mean(roberta_counts):.1f}, median: {np.median(roberta_counts):.0f}, "
      f"max: {max(roberta_counts)}, >512: {sum(c > 512 for c in roberta_counts)} ({sum(c > 512 for c in roberta_counts)/len(roberta_counts)*100:.1f}%)")
print(f"ModernBERT — mean: {np.mean(modernbert_counts):.1f}, median: {np.median(modernbert_counts):.0f}, "
      f"max: {max(modernbert_counts)}, >512: {sum(c > 512 for c in modernbert_counts)} ({sum(c > 512 for c in modernbert_counts)/len(modernbert_counts)*100:.1f}%)")

In [ ]:
# Token count distribution histogram
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Token Count Distribution — RoBERTa vs ModernBERT on dair-ai/emotion",
             fontsize=12, fontweight="bold")

for ax, counts, name, color in [
    (axes[0], roberta_counts,    "RoBERTa",    "#4e79a7"),
    (axes[1], modernbert_counts, "ModernBERT", "#e15759")
]:
    ax.hist(counts, bins=30, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(np.mean(counts), color="black", linestyle="--",
               linewidth=1.5, label=f"Mean: {np.mean(counts):.1f}")
    ax.axvline(np.median(counts), color="gray", linestyle=":",
               linewidth=1.5, label=f"Median: {np.median(counts):.0f}")
    ax.set_title(f"{name} Token Counts", fontsize=11, fontweight="bold")
    ax.set_xlabel("Token Count")
    ax.set_ylabel("Frequency")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("../results/figures/tokenization_comparison.png", dpi=150)
plt.show()

# Show tokenization of example texts
example_texts = [
    "i feel so lost and miserable :(",
    "omg i cant believe this is happening rn!! 😱",
    "feeling gr8 today absolutely blessed"
]

print("\nTokenization Examples:")
print("=" * 80)
for text in example_texts:
    rob_toks = roberta_tokenizer.tokenize(text)
    mod_toks = modernbert_tokenizer.tokenize(text)
    print(f"\nText: '{text}'")
    print(f"  RoBERTa    ({len(rob_toks)} tokens): {rob_toks}")
    print(f"  ModernBERT ({len(mod_toks)} tokens): {mod_toks}")

## Section 7: HuggingFace Hub Push

### Sharing your fine-tuned model

Once you've trained and calibrated your model, you can push it to the HuggingFace Hub for sharing and serving. The `push_to_hub=True` flag in `TextTrainingConfig` will:

1. Create or update the model repository on the Hub
2. Upload model weights, tokenizer, and configuration
3. Auto-generate a model card with training details and evaluation metrics
4. Make the model available for inference via the Inference API

**Before running**: Replace `YOUR_HF_USERNAME` with your actual HuggingFace username and ensure you're logged in with `huggingface-cli login`.

In [ ]:
# Push to HuggingFace Hub — uncomment and fill in your username to run

# cfg = TextTrainingConfig(
#     model_key="modernbert",
#     push_to_hub=True,
#     hub_model_id="YOUR_HF_USERNAME/eurosat-emotion-modernbert"
# )
# results = train_text_model(cfg)

# To push an already-trained model:
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path="../results/text/modernbert",
#     repo_id="YOUR_HF_USERNAME/eurosat-emotion-modernbert",
#     repo_type="model"
# )

print("Push to Hub cell — ready to run after filling in YOUR_HF_USERNAME.")
print("Run: huggingface-cli login  (to authenticate)")